# Worldwide Earthquake Events API - Silver Layer Processing

In [1]:
from pyspark.sql.functions import col
from pyspark.sql.types import TimestampType

StatementMeta(, 97ec7973-0902-4e8f-bc9a-90b1f843a452, 3, Finished, Available, Finished, False)

In [2]:
from datetime import date, timedelta
start_date = date.today() - timedelta(7)
print(start_date)

StatementMeta(, 97ec7973-0902-4e8f-bc9a-90b1f843a452, 4, Finished, Available, Finished, False)

2026-02-27


In [3]:
# df now is a Spark DataFrame containing JSON data
df = spark.read.option("multiline", "true").json(f"Files/{start_date}_earthquake_data.json")

StatementMeta(, 97ec7973-0902-4e8f-bc9a-90b1f843a452, 5, Finished, Available, Finished, False)

In [8]:
# Reshape earthquake data by extracting and renaming key attributes for further analysis.
df = \
df.\
    select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'),
        col('geometry.coordinates').getItem(1).alias('latitude'),
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias('updated')
        )

StatementMeta(, 97ec7973-0902-4e8f-bc9a-90b1f843a452, 10, Finished, Available, Finished, False)

In [9]:
display(df)

StatementMeta(, 97ec7973-0902-4e8f-bc9a-90b1f843a452, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4e381766-ae06-468f-ab8d-049406688ea5)

In [10]:
# Convert 'time' and 'updated' columns from milliseconds to timestamp format for clearer datetime representation.
df = df.\
    withColumn('time', col('time')/1000).\
    withColumn('updated', col('updated')/1000).\
    withColumn('time', col('time').cast(TimestampType())).\
    withColumn('updated', col('updated').cast(TimestampType()))

StatementMeta(, 97ec7973-0902-4e8f-bc9a-90b1f843a452, 12, Finished, Available, Finished, False)

In [11]:
display(df)

StatementMeta(, 97ec7973-0902-4e8f-bc9a-90b1f843a452, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0d82e7c4-cbb0-4145-a436-c368acbe2163)

In [12]:
# appending the data to the gold table
df.write.mode('append').saveAsTable('earthquake_events_silver')

StatementMeta(, 97ec7973-0902-4e8f-bc9a-90b1f843a452, 14, Finished, Available, Finished, False)